# 07 Class Weight Optimization

对比无权重、自动平衡权重、手动高风险放大权重三组 LightGBM 风险分类模型，并输出最优类别权重配置。


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for candidate in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    candidate_text = str(candidate)
    if candidate_text not in sys.path:
        sys.path.insert(0, candidate_text)

from src.features.feature_selector import CreditFeatureSelector
from src.features.preprocessor import CreditDataPreprocessor
from src.models.model_evaluator import CreditModelEvaluator
from src.models.risk_classifier import CreditRiskClassifier

DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
MODEL_DIR = PROJECT_ROOT / "src" / "models"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["正常类", "关注类", "次级类", "可疑类", "损失类"]
MANUAL_WEIGHT_TEMPLATE = {0: 1, 1: 5, 2: 20, 3: 35}

def resolve_split_path(split_name: str) -> Path:
    purified_path = DATA_DIR / f"purified_{split_name}.csv"
    return purified_path if purified_path.exists() else DATA_DIR / f"{split_name}.csv"


In [ ]:
train_df = pd.read_csv(resolve_split_path("train"), low_memory=False)
val_df = pd.read_csv(resolve_split_path("val"), low_memory=False)
test_df = pd.read_csv(resolve_split_path("test"), low_memory=False)

y_train = train_df["preloan_risk_label"].astype(int)
y_val = val_df["preloan_risk_label"].astype(int)
y_test = test_df["preloan_risk_label"].astype(int)

preprocessor = CreditDataPreprocessor(target_column="preloan_risk_label")
X_train_processed = preprocessor.fit_transform(train_df)
X_val_processed = preprocessor.transform(val_df)
X_test_processed = preprocessor.transform(test_df)

selector = CreditFeatureSelector(
    top_k_features=80,
    use_pca=False,
    n_estimators=100,
)
X_train_selected = selector.fit_transform(X_train_processed, y_train)
X_val_selected = selector.transform(X_val_processed)
X_test_selected = selector.transform(X_test_processed)

active_labels = sorted(y_train.unique().tolist())
active_manual_weights = {
    label: MANUAL_WEIGHT_TEMPLATE.get(int(label), 1.0)
    for label in active_labels
}
active_class_names = [CLASS_NAMES[label] for label in active_labels if label < len(CLASS_NAMES)]

display(Markdown(f"当前训练集中实际出现的类别标签: `{active_labels}`"))
display(Markdown(f"当前实验生效的手动权重: `{active_manual_weights}`"))


In [ ]:
def compute_high_risk_recall(metric_payload: dict) -> float:
    report = metric_payload["classification_report_named"]
    risk_recalls = [
        float(report.get(class_name, {}).get("recall", 0.0))
        for class_name in metric_payload["class_names"]
        if class_name != "正常类"
    ]
    if not risk_recalls:
        return 0.0
    return float(sum(risk_recalls) / len(risk_recalls))


def evaluate_experiment(experiment_name: str, class_weight_mode: str, custom_weight: dict[int, float] | None):
    classifier = CreditRiskClassifier(
        model_type="lightgbm",
        class_weight_mode=class_weight_mode,
        custom_class_weight=custom_weight,
    )
    classifier.fit(X_train_selected, y_train)

    val_evaluator = CreditModelEvaluator(
        model=classifier,
        X_test=X_val_selected,
        y_test=y_val,
        class_names=CLASS_NAMES,
    )
    val_metrics = val_evaluator.evaluate_imbalanced_multiclass()

    test_evaluator = CreditModelEvaluator(
        model=classifier,
        X_test=X_test_selected,
        y_test=y_test,
        class_names=CLASS_NAMES,
    )
    test_metrics = test_evaluator.evaluate_imbalanced_multiclass()

    return {
        "experiment_name": experiment_name,
        "class_weight_mode": class_weight_mode,
        "resolved_class_weight": classifier.get_class_weight_distribution()["resolved_class_weight"],
        "val_macro_f1": float(val_metrics["macro_f1"]),
        "val_weighted_f1": float(val_metrics["weighted_f1"]),
        "val_high_risk_recall": compute_high_risk_recall(val_metrics),
        "test_macro_f1": float(test_metrics["macro_f1"]),
        "test_weighted_f1": float(test_metrics["weighted_f1"]),
        "test_high_risk_recall": compute_high_risk_recall(test_metrics),
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


In [ ]:
experiment_results = []
experiment_results.append(
    evaluate_experiment(
        experiment_name="baseline_no_weight",
        class_weight_mode="none",
        custom_weight=None,
    )
)
experiment_results.append(
    evaluate_experiment(
        experiment_name="auto_balanced_weight",
        class_weight_mode="auto",
        custom_weight=None,
    )
)
experiment_results.append(
    evaluate_experiment(
        experiment_name="manual_high_risk_weight",
        class_weight_mode="manual",
        custom_weight=active_manual_weights,
    )
)

comparison_df = pd.DataFrame(
    [
        {
            "实验组": item["experiment_name"],
            "权重模式": item["class_weight_mode"],
            "验证集 Macro-F1": round(item["val_macro_f1"], 6),
            "验证集 Weighted-F1": round(item["val_weighted_f1"], 6),
            "验证集高风险召回率": round(item["val_high_risk_recall"], 6),
            "测试集 Macro-F1": round(item["test_macro_f1"], 6),
            "测试集 Weighted-F1": round(item["test_weighted_f1"], 6),
            "测试集高风险召回率": round(item["test_high_risk_recall"], 6),
            "实际权重": json.dumps(item["resolved_class_weight"], ensure_ascii=False),
        }
        for item in experiment_results
    ]
)
display(comparison_df)


In [ ]:
plot_frame = comparison_df[["实验组", "验证集 Macro-F1", "验证集高风险召回率"]].copy()
positions = range(len(plot_frame))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(
    [position - bar_width / 2 for position in positions],
    plot_frame["验证集 Macro-F1"],
    width=bar_width,
    label="验证集 Macro-F1",
)
ax.bar(
    [position + bar_width / 2 for position in positions],
    plot_frame["验证集高风险召回率"],
    width=bar_width,
    label="验证集高风险召回率",
)
ax.set_xticks(list(positions))
ax.set_xticklabels(plot_frame["实验组"], rotation=15, ha="right")
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Score")
ax.set_title("类别权重优化对比")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.3)

output_path = FIGURE_DIR / "weight_optimization_comparison.png"
fig.tight_layout()
fig.savefig(output_path, dpi=200, bbox_inches="tight")
plt.close(fig)

display(Markdown(f"对比图已保存到 `results/figures/{output_path.name}`"))


In [ ]:
best_result = sorted(
    experiment_results,
    key=lambda item: (
        item["val_macro_f1"],
        item["val_high_risk_recall"],
        item["test_macro_f1"],
    ),
    reverse=True,
)[0]

best_payload = {
    "selected_by": "validation_macro_f1_then_high_risk_recall_then_test_macro_f1",
    "experiment_name": best_result["experiment_name"],
    "class_weight_mode": best_result["class_weight_mode"],
    "resolved_class_weight": best_result["resolved_class_weight"],
    "active_class_labels": active_labels,
    "active_class_names": best_result["val_metrics"]["class_names"],
    "manual_weight_template": MANUAL_WEIGHT_TEMPLATE,
    "metrics": {
        "val_macro_f1": best_result["val_macro_f1"],
        "val_weighted_f1": best_result["val_weighted_f1"],
        "val_high_risk_recall": best_result["val_high_risk_recall"],
        "test_macro_f1": best_result["test_macro_f1"],
        "test_weighted_f1": best_result["test_weighted_f1"],
        "test_high_risk_recall": best_result["test_high_risk_recall"],
    },
}

best_config_path = MODEL_DIR / "best_class_weight_config.json"
best_config_path.write_text(
    json.dumps(best_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(Markdown(f"最优权重配置已保存到 `src/models/{best_config_path.name}`"))
display(Markdown(json.dumps(best_payload, ensure_ascii=False, indent=2)))
